In [15]:
# Bloco 1 - Importar a base de motoristas com cidade e estado

from pathlib import Path
import pandas as pd


def obter_pasta_do_notebook():
    return Path.cwd()


def importar_csv_mesma_pasta(nome_arquivo, nome_base, colunas=None, sep=",", engine=None):
    try:

        pasta_notebook = obter_pasta_do_notebook()
        caminho_arquivo = pasta_notebook / nome_arquivo

        argumentos = {
            "filepath_or_buffer": caminho_arquivo,
            "sep": sep,
            "encoding": "utf-8-sig",
            "usecols": colunas
        }

        if engine is not None:
            argumentos["engine"] = engine
        else:
            argumentos["low_memory"] = False

        df = pd.read_csv(**argumentos)

        df.columns = df.columns.str.strip()

        print(f"Base '{nome_base}' importada com sucesso!")
        print(f"Arquivo importado: {caminho_arquivo}")
        print(f"Quantidade de linhas: {df.shape[0]}")
        print(f"Quantidade de colunas: {df.shape[1]}")
        print()

        return df

    except FileNotFoundError:

        print(f"Erro: arquivo da base '{nome_base}' não encontrado.")
        print(f"Verifique se o arquivo '{nome_arquivo}' está na mesma pasta do notebook.")

        return None

    except Exception as erro:

        print(f"Erro ao importar a base '{nome_base}'.")
        print(f"Detalhes do erro: {erro}")

        return None


def inserir_cidade_estado(df_motoristas, df_municipios):

    df_motoristas = df_motoristas.copy()
    df_municipios = df_municipios.copy()

    df_motoristas.columns = df_motoristas.columns.str.strip()
    df_municipios.columns = df_municipios.columns.str.strip()

    colunas_municipios_necessarias = [
        "Código Município Completo",
        "Nome_Município",
        "Nome_UF"
    ]

    colunas_faltantes = [
        coluna
        for coluna in colunas_municipios_necessarias
        if coluna not in df_municipios.columns
    ]

    if colunas_faltantes:

        print("Erro: algumas colunas necessárias não foram encontradas na base de municípios.")
        print(colunas_faltantes)

        return None

    df_motoristas["codigo_cidade"] = pd.to_numeric(
        df_motoristas["codigo_cidade"],
        errors="coerce"
    ).astype("Int64")

    df_municipios["Código Município Completo"] = pd.to_numeric(
        df_municipios["Código Município Completo"],
        errors="coerce"
    ).astype("Int64")

    municipios = (

        df_municipios[
            [
                "Código Município Completo",
                "Nome_Município",
                "Nome_UF"
            ]
        ]

        .drop_duplicates(subset=["Código Município Completo"])

        .rename(
            columns={
                "Código Município Completo": "codigo_cidade",
                "Nome_Município": "nome_cidade",
                "Nome_UF": "estado"
            }
        )

    )

    df_motoristas = df_motoristas.merge(
        municipios,
        on="codigo_cidade",
        how="left"
    )

    colunas = list(df_motoristas.columns)

    colunas.remove("nome_cidade")
    colunas.remove("estado")

    posicao = colunas.index("codigo_cidade")

    colunas.insert(posicao + 1, "nome_cidade")
    colunas.insert(posicao + 2, "estado")

    df_motoristas = df_motoristas[colunas]

    print("Cidade e estado inseridos com sucesso!")

    print(
        "Motoristas sem município identificado:",
        df_motoristas["nome_cidade"].isna().sum()
    )

    return df_motoristas


nome_arquivo_motoristas = "script_06_motoristas.csv"
nome_arquivo_municipios = "Lista de Municipios IBGE.csv"

colunas_motoristas = [
    "codigo_cidade",
    "idade",
    "sexo",
    "tem_cnh_a",
    "tem_cnh_b",
    "tem_cnh_c",
    "tem_cnh_d",
    "tem_cnh_e",
    "turno_manha",
    "turno_tarde",
    "turno_noite",
    "data_validade_cnh",
    "vinculo",
    "salario"
]

df_motoristas = importar_csv_mesma_pasta(
    nome_arquivo_motoristas,
    "motoristas",
    colunas=colunas_motoristas,
    sep=","
)

df_municipios = importar_csv_mesma_pasta(
    nome_arquivo_municipios,
    "municípios",
    sep=None,
    engine="python"
)

if df_motoristas is not None and df_municipios is not None:

    df_motoristas_transformado = inserir_cidade_estado(
        df_motoristas,
        df_municipios
    )

    if df_motoristas_transformado is not None:

        display(df_motoristas_transformado)

Base 'motoristas' importada com sucesso!
Arquivo importado: c:\Users\mathe\OneDrive\Trabalho\00-UFG\Transporte escolar\Frente de indicadores\Dados SETE 2026\Dados dados sete consolidados\script_06_motoristas.csv
Quantidade de linhas: 63739
Quantidade de colunas: 14

Base 'municípios' importada com sucesso!
Arquivo importado: c:\Users\mathe\OneDrive\Trabalho\00-UFG\Transporte escolar\Frente de indicadores\Dados SETE 2026\Dados dados sete consolidados\Lista de Municipios IBGE.csv
Quantidade de linhas: 5572
Quantidade de colunas: 13

Cidade e estado inseridos com sucesso!
Motoristas sem município identificado: 0


,codigo_cidade,nome_cidade,estado,idade,sexo,tem_cnh_a,tem_cnh_b,tem_cnh_c,tem_cnh_d,tem_cnh_e,turno_manha,turno_tarde,turno_noite,data_validade_cnh,vinculo,salario
0,1304401,Urucurituba,Amazonas,31,1,S,N,N,N,N,S,N,N,2031-03-04,NaN,NaN
1,3170479,Uruana de Minas,Minas Gerais,64,1,N,N,N,S,N,S,S,N,2027-03-16,4.0,NaN
2,2928703,Santo Antônio de Jesus,Bahia,45,1,S,N,N,S,N,S,S,N,2034-12-16,3.0,NaN
3,3152105,Ponte Nova,Minas Gerais,50,1,S,N,N,S,N,S,S,N,2029-06-28,4.0,NaN
4,2208007,Picos,Piauí,57,1,S,N,N,S,N,S,S,N,2028-05-08,4.0,2500.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
63734,4124053,Santa Terezinha de Itaipu,Paraná,57,1,N,N,N,S,N,S,S,N,2029-05-29,3.0,2440.0
63735,4124053,Santa Terezinha de Itaipu,Paraná,66,1,N,N,N,S,N,N,S,N,2026-03-24,3.0,2440.0
63736,2104800,Grajaú,Maranhão,55,1,N,N,N,S,N,S,S,S,2029-12-08,1.0,NaN
63737,2310308,Parambu,Ceará,38,1,N,N,N,S,N,S,N,N,2027-03-12,NaN,NaN


In [16]:
# Bloco 2 - Preparar a base de motoristas para consolidação

df_consolidacao = df_motoristas_transformado.copy()

# Preencher estados não identificados
df_consolidacao["estado"] = (
    df_consolidacao["estado"]
    .fillna("Não identificado")
)

# Converter campos numéricos
colunas_numericas = [
    "sexo",
    "vinculo",
    "salario"
]

for coluna in colunas_numericas:

    df_consolidacao[coluna] = pd.to_numeric(
        df_consolidacao[coluna],
        errors="coerce"
    )

# Padronizar campos Sim/Não

colunas_sim_nao = [
    "tem_cnh_a",
    "tem_cnh_b",
    "tem_cnh_c",
    "tem_cnh_d",
    "tem_cnh_e",
    "turno_manha",
    "turno_tarde",
    "turno_noite"
]

for coluna in colunas_sim_nao:

    df_consolidacao[coluna] = (
        df_consolidacao[coluna]
        .fillna("N")
        .astype(str)
        .str.strip()
        .str.upper()
    )

print("Base de motoristas preparada para consolidação.")

display(df_consolidacao.head())

Base de motoristas preparada para consolidação.


,codigo_cidade,nome_cidade,estado,idade,sexo,tem_cnh_a,tem_cnh_b,tem_cnh_c,tem_cnh_d,tem_cnh_e,turno_manha,turno_tarde,turno_noite,data_validade_cnh,vinculo,salario
0,1304401,Urucurituba,Amazonas,31,1,S,N,N,N,N,S,N,N,2031-03-04,NaN,NaN
1,3170479,Uruana de Minas,Minas Gerais,64,1,N,N,N,S,N,S,S,N,2027-03-16,4.0,NaN
2,2928703,Santo Antônio de Jesus,Bahia,45,1,S,N,N,S,N,S,S,N,2034-12-16,3.0,NaN
3,3152105,Ponte Nova,Minas Gerais,50,1,S,N,N,S,N,S,S,N,2029-06-28,4.0,NaN
4,2208007,Picos,Piauí,57,1,S,N,N,S,N,S,S,N,2028-05-08,4.0,2500.0


In [17]:
# Bloco 3 - Função para consolidar vínculo por estado

def consolidar_vinculo_por_estado(df):

    tabela = (
        df
        .groupby("estado")
        .agg(
            total_motoristas=("codigo_cidade", "size"),
            servidor_efetivo=("vinculo", lambda x: (x == 1).sum()),
            servidor_comissionado=("vinculo", lambda x: (x == 2).sum()),
            terceirizado=("vinculo", lambda x: (x == 3).sum()),
            outro_vinculo=("vinculo", lambda x: (x == 4).sum()),
            vinculo_nao_identificado=("vinculo", lambda x: (~x.isin([1, 2, 3, 4])).sum())
        )
        .reset_index()
        .sort_values("estado")
    )

    print("Tabela consolidada de vínculo por estado:")

    display(tabela)

    return tabela

In [18]:
# Bloco 4 - Consolidar vínculo

tabela_vinculo = consolidar_vinculo_por_estado(
    df_consolidacao
)

Tabela consolidada de vínculo por estado:


,estado,total_motoristas,servidor_efetivo,servidor_comissionado,terceirizado,outro_vinculo,vinculo_nao_identificado
0,Acre,124,44,2,45,12,21
1,Alagoas,1844,164,54,401,616,609
2,Amapá,82,3,6,24,19,30
3,Amazonas,291,20,10,98,53,110
4,Bahia,6488,329,252,2574,853,2480
5,Ceará,3294,410,166,1194,602,922
6,Espírito Santo,890,58,37,374,26,395
7,Goiás,1731,383,248,653,62,385
8,Maranhão,1903,120,63,1004,311,405
9,Mato Grosso,1990,648,96,524,352,370


In [19]:
# Bloco 6 - Consolidar categorias da CNH por estado

colunas_cnh = [
    "tem_cnh_a",
    "tem_cnh_b",
    "tem_cnh_c",
    "tem_cnh_d",
    "tem_cnh_e"
]

# Padronizar os valores
for coluna in colunas_cnh:

    df_consolidacao[coluna] = (
        df_consolidacao[coluna]
        .fillna("N")
        .astype(str)
        .str.strip()
        .str.upper()
    )

tabela_cnh = (
    df_consolidacao
    .groupby("estado")
    .agg(
        total_motoristas=("codigo_cidade", "size"),

        possui_cnh_a=("tem_cnh_a", lambda x: (x == "S").sum()),
        nao_possui_cnh_a=("tem_cnh_a", lambda x: (x != "S").sum()),

        possui_cnh_b=("tem_cnh_b", lambda x: (x == "S").sum()),
        nao_possui_cnh_b=("tem_cnh_b", lambda x: (x != "S").sum()),

        possui_cnh_c=("tem_cnh_c", lambda x: (x == "S").sum()),
        nao_possui_cnh_c=("tem_cnh_c", lambda x: (x != "S").sum()),

        possui_cnh_d=("tem_cnh_d", lambda x: (x == "S").sum()),
        nao_possui_cnh_d=("tem_cnh_d", lambda x: (x != "S").sum()),

        possui_cnh_e=("tem_cnh_e", lambda x: (x == "S").sum()),
        nao_possui_cnh_e=("tem_cnh_e", lambda x: (x != "S").sum())
    )
    .reset_index()
    .sort_values("estado")
)

print("Tabela consolidada das categorias da CNH por estado:")

display(tabela_cnh)

Tabela consolidada das categorias da CNH por estado:


,estado,total_motoristas,possui_cnh_a,nao_possui_cnh_a,possui_cnh_b,nao_possui_cnh_b,possui_cnh_c,nao_possui_cnh_c,possui_cnh_d,nao_possui_cnh_d,possui_cnh_e,nao_possui_cnh_e
0,Acre,124,64,60,11,113,4,120,102,22,15,109
1,Alagoas,1844,1208,636,299,1545,221,1623,1539,305,253,1591
2,Amapá,82,60,22,21,61,16,66,49,33,6,76
3,Amazonas,291,233,58,70,221,17,274,99,192,6,285
4,Bahia,6488,4389,2099,1556,4932,448,6040,4917,1571,453,6035
5,Ceará,3294,2520,774,454,2840,358,2936,3081,213,139,3155
6,Espírito Santo,890,596,294,144,746,138,752,750,140,174,716
7,Goiás,1731,1110,621,103,1628,79,1652,1430,301,289,1442
8,Maranhão,1903,1199,704,380,1523,66,1837,1403,500,145,1758
9,Mato Grosso,1990,1483,507,434,1556,426,1564,1559,431,510,1480


In [20]:
# Bloco 6 - Consolidar categorias da CNH por estado

colunas_cnh = [
    "tem_cnh_a",
    "tem_cnh_b",
    "tem_cnh_c",
    "tem_cnh_d",
    "tem_cnh_e"
]

# Padronizar os valores
for coluna in colunas_cnh:

    df_consolidacao[coluna] = (
        df_consolidacao[coluna]
        .fillna("N")
        .astype(str)
        .str.strip()
        .str.upper()
    )

tabela_cnh = (
    df_consolidacao
    .groupby("estado")
    .agg(
        total_motoristas=("codigo_cidade", "size"),

        possui_cnh_a=("tem_cnh_a", lambda x: (x == "S").sum()),
        nao_possui_cnh_a=("tem_cnh_a", lambda x: (x != "S").sum()),

        possui_cnh_b=("tem_cnh_b", lambda x: (x == "S").sum()),
        nao_possui_cnh_b=("tem_cnh_b", lambda x: (x != "S").sum()),

        possui_cnh_c=("tem_cnh_c", lambda x: (x == "S").sum()),
        nao_possui_cnh_c=("tem_cnh_c", lambda x: (x != "S").sum()),

        possui_cnh_d=("tem_cnh_d", lambda x: (x == "S").sum()),
        nao_possui_cnh_d=("tem_cnh_d", lambda x: (x != "S").sum()),

        possui_cnh_e=("tem_cnh_e", lambda x: (x == "S").sum()),
        nao_possui_cnh_e=("tem_cnh_e", lambda x: (x != "S").sum())
    )
    .reset_index()
    .sort_values("estado")
)

print("Tabela consolidada das categorias da CNH por estado:")

display(tabela_cnh)

Tabela consolidada das categorias da CNH por estado:


,estado,total_motoristas,possui_cnh_a,nao_possui_cnh_a,possui_cnh_b,nao_possui_cnh_b,possui_cnh_c,nao_possui_cnh_c,possui_cnh_d,nao_possui_cnh_d,possui_cnh_e,nao_possui_cnh_e
0,Acre,124,64,60,11,113,4,120,102,22,15,109
1,Alagoas,1844,1208,636,299,1545,221,1623,1539,305,253,1591
2,Amapá,82,60,22,21,61,16,66,49,33,6,76
3,Amazonas,291,233,58,70,221,17,274,99,192,6,285
4,Bahia,6488,4389,2099,1556,4932,448,6040,4917,1571,453,6035
5,Ceará,3294,2520,774,454,2840,358,2936,3081,213,139,3155
6,Espírito Santo,890,596,294,144,746,138,752,750,140,174,716
7,Goiás,1731,1110,621,103,1628,79,1652,1430,301,289,1442
8,Maranhão,1903,1199,704,380,1523,66,1837,1403,500,145,1758
9,Mato Grosso,1990,1483,507,434,1556,426,1564,1559,431,510,1480


In [21]:
# Bloco 7 - Consolidar turnos por estado

tabela_turnos = (
    df_consolidacao
    .groupby("estado")
    .agg(
        total_motoristas=("codigo_cidade", "size"),
        turno_manha=("turno_manha", lambda x: (x == "S").sum()),
        turno_tarde=("turno_tarde", lambda x: (x == "S").sum()),
        turno_noite=("turno_noite", lambda x: (x == "S").sum())
    )
    .reset_index()
    .sort_values("estado")
)

print("Tabela consolidada dos turnos por estado:")

display(tabela_turnos)

Tabela consolidada dos turnos por estado:


,estado,total_motoristas,turno_manha,turno_tarde,turno_noite
0,Acre,124,119,95,8
1,Alagoas,1844,1612,1630,652
2,Amapá,82,76,76,12
3,Amazonas,291,240,181,23
4,Bahia,6488,5342,4648,1319
5,Ceará,3294,3094,3056,279
6,Espírito Santo,890,837,693,44
7,Goiás,1731,1358,1254,259
8,Maranhão,1903,1519,1629,300
9,Mato Grosso,1990,1484,1437,138


In [22]:
# Bloco 8 - Consolidar todas as tabelas e exportar para Excel

from functools import reduce


def consolidar_tabelas_por_estado(tabelas):

    tabelas_preparadas = []

    for indice, tabela in enumerate(tabelas):

        tabela_temp = tabela.copy()

        if indice > 0 and "total_motoristas" in tabela_temp.columns:
            tabela_temp = tabela_temp.drop(columns=["total_motoristas"])

        tabelas_preparadas.append(tabela_temp)

    tabela_final = reduce(
        lambda esquerda, direita: esquerda.merge(
            direita,
            on="estado",
            how="outer"
        ),
        tabelas_preparadas
    )

    tabela_final = (
        tabela_final
        .sort_values("estado")
        .reset_index(drop=True)
    )

    # Consolidação Brasil

    linha_brasil = {
        "estado": "Brasil"
    }

    for coluna in tabela_final.columns:

        if coluna != "estado":
            linha_brasil[coluna] = tabela_final[coluna].sum()

    tabela_final = pd.concat(
        [
            tabela_final,
            pd.DataFrame([linha_brasil])
        ],
        ignore_index=True
    )

    return tabela_final


tabelas_para_consolidar = [
    tabela_vinculo,
    tabela_cnh,
    tabela_turnos
]

tabela_consolidada_final = consolidar_tabelas_por_estado(
    tabelas_para_consolidar
)

pasta_notebook = obter_pasta_do_notebook()

nome_arquivo_excel = "Consolidação 06 - script_06_motoristas.xlsx"

caminho_arquivo_excel = pasta_notebook / nome_arquivo_excel

tabela_consolidada_final.to_excel(
    caminho_arquivo_excel,
    index=False,
    sheet_name="consolidado"
)

print("Tabela consolidada final exportada com sucesso!")
print(f"Arquivo Excel exportado: {caminho_arquivo_excel}")

display(tabela_consolidada_final)

Tabela consolidada final exportada com sucesso!
Arquivo Excel exportado: c:\Users\mathe\OneDrive\Trabalho\00-UFG\Transporte escolar\Frente de indicadores\Dados SETE 2026\Dados dados sete consolidados\Consolidação 06 - script_06_motoristas.xlsx


,estado,total_motoristas,servidor_efetivo,servidor_comissionado,terceirizado,outro_vinculo,vinculo_nao_identificado,possui_cnh_a,nao_possui_cnh_a,possui_cnh_b,nao_possui_cnh_b,possui_cnh_c,nao_possui_cnh_c,possui_cnh_d,nao_possui_cnh_d,possui_cnh_e,nao_possui_cnh_e,turno_manha,turno_tarde,turno_noite
0,Acre,124,44,2,45,12,21,64,60,11,113,4,120,102,22,15,109,119,95,8
1,Alagoas,1844,164,54,401,616,609,1208,636,299,1545,221,1623,1539,305,253,1591,1612,1630,652
2,Amapá,82,3,6,24,19,30,60,22,21,61,16,66,49,33,6,76,76,76,12
3,Amazonas,291,20,10,98,53,110,233,58,70,221,17,274,99,192,6,285,240,181,23
4,Bahia,6488,329,252,2574,853,2480,4389,2099,1556,4932,448,6040,4917,1571,453,6035,5342,4648,1319
5,Ceará,3294,410,166,1194,602,922,2520,774,454,2840,358,2936,3081,213,139,3155,3094,3056,279
6,Espírito Santo,890,58,37,374,26,395,596,294,144,746,138,752,750,140,174,716,837,693,44
7,Goiás,1731,383,248,653,62,385,1110,621,103,1628,79,1652,1430,301,289,1442,1358,1254,259
8,Maranhão,1903,120,63,1004,311,405,1199,704,380,1523,66,1837,1403,500,145,1758,1519,1629,300
9,Mato Grosso,1990,648,96,524,352,370,1483,507,434,1556,426,1564,1559,431,510,1480,1484,1437,138
